# 09 · Experiments & Exploratory Work

**Purpose.** Preserves exploratory, superseded, or abandoned code exactly as it was written. Nothing in this notebook feeds the final results in notebooks 01–08, but it documents the modeling journey and the alternatives that were tried and set aside — useful context for future students continuing this work.

Cells are grouped by sub-theme and kept in their original chronological order within each group.


## Sweep-plot early versions

Superseded by the tick/style-improved versions kept in notebook 03 (same underlying CSVs and logic).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV file
df = pd.read_csv("up and down, 0.5%.csv")

# Extract up and down sweep data
duty_up = df["Duty Cycle (%)"]
rpm_up = df["RPM"]

duty_down = df["Duty cycle"]
rpm_down = df["RPM.1"]

# Plot both sweeps
plt.figure(figsize=(10, 5))
plt.plot(duty_up, rpm_up, label='Up Sweep', color='blue', marker='o')
plt.plot(duty_down, rpm_down, label='Down Sweep', color='red', marker='x')
plt.xlabel("Duty Cycle (%)")
plt.ylabel("RPM")
plt.title("Duty Cycle vs RPM for 20kHz")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV (adjust path)
df = pd.read_csv("duty_freq_rpm_data_updown_25khz.csv", header=0)

# Check actual columns
print("Original columns:", df.columns.tolist())

# Drop the unexpected or unnamed column if it's just empty
df = df.dropna(axis=1, how='all')

# Rename correctly
df.columns = ['Duty_Up', 'Freq_Up', 'RPM_Up', 'Duty_Down', 'Freq_Down', 'RPM_Down']

# Clean up empty rows
df.dropna(how='all', inplace=True)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(df['Duty_Up'], df['RPM_Up'], label='Up Sweep', marker='o', color='blue')
plt.plot(df['Duty_Down'], df['RPM_Down'], label='Down Sweep', marker='x', color='red')

plt.xlabel("Duty Cycle (%)")
plt.ylabel("RPM")
plt.title("Duty Cycle vs RPM for 25 kHz")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Voltage-to-duty mapping — early exploration

Linear interpolation, incremental averaging steps, and duplicate interpolation attempts that preceded the final approach in notebook 05.

In [ ]:
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

# Given multimeter voltage vs duty cycle data (z=2, r=0)
duty_cycle = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50,
                       55, 60, 65, 70, 75, 80, 85, 90, 95, 100])
voltage = np.array([0.826, 0.8166, 0.8066, 0.8033, 0.8, 0.796,
                    0.793, 0.79, 0.776, 0.763, 0.743, 0.733,
                    0.7, 0.666, 0.641, 0.616, 0.6, 0.586, 0.643, 0.623])

# Fit an interpolation model: voltage -> duty cycle
voltage_to_duty = interp1d(voltage, duty_cycle, kind='linear', fill_value="extrapolate")

# Example: Estimate duty cycle from new voltages
new_voltages = np.array([0.79, 0.763, 0.7, 0.6])
estimated_duties = voltage_to_duty(new_voltages)

# Print results
for v, d in zip(new_voltages, estimated_duties):
    print(f"Voltage: {v:.3f} V -> Estimated Duty Cycle: {d:.2f} %")

# Optional: Plot the mapping
plt.plot(voltage, duty_cycle, 'o-', label='Voltage → Duty Mapping')
plt.xlabel("Voltage (V) at z=2, r=0")
plt.ylabel("Duty Cycle (%)")
plt.grid(True)
plt.title("Interpolated Voltage to Duty Cycle Mapping")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the Excel file
file_path = "z=2,r=0.xlsx"
df = pd.read_excel(file_path, sheet_name='21')

# Clean column names
df.columns = df.columns.str.strip()

# Drop any rows with missing duty or voltage
df = df.dropna(subset=['Duty', 'Voltage'])

# Group by Duty cycle and compute mean voltage
grouped = df.groupby('Duty')['Voltage'].mean().reset_index()

# Plot Duty Cycle vs Voltage
plt.figure(figsize=(10, 5))
plt.plot(grouped['Duty'], grouped['Voltage'], marker='o', color='purple')
plt.xlabel('Duty Cycle (%)')
plt.ylabel('Average Voltage (V)')
plt.title('Duty Cycle vs Voltage for z = 2, r = 0')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load Excel file and correct sheet
file_path = "z=2,r=0.xlsx"
df = pd.read_excel(file_path, sheet_name='21')

# Clean up column names
df.columns = df.columns.str.strip()

# Drop rows with missing values in relevant columns
df = df.dropna(subset=['Duty', 'Voltage'])

# Group by duty cycle and compute mean voltage for each
grouped = df.groupby('Duty')['Voltage'].mean().reset_index()

# Sort just in case
grouped = grouped.sort_values(by='Duty')

In [ ]:
# Set up detailed ticks
xticks = np.arange(0, 102, 2)          # Every 2% duty cycle
yticks = np.arange(0, 2, 0.2)        # Every 0.2V from 0V to 3.5V

# Plot
plt.figure(figsize=(12, 6))
plt.plot(grouped['Duty'], grouped['Voltage'], marker='o', linestyle='-', color='purple')
plt.xticks(xticks)
plt.yticks(yticks)
plt.xlabel('Duty Cycle (%)')
plt.ylabel('Average Voltage (V)')
plt.title('Duty Cycle vs Average Voltage (z = 2, r = 0)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# Measured values from your data
voltages = np.array([1.00235314, 0.82740628, 0.81370782, 0.81502292, 0.81771553,
                     0.8102661, 0.80432675, 0.79196567, 0.77636817, 0.76105513,
                     0.74830033, 0.73875982, 0.72980107, 0.72330292, 0.71230216,
                     0.71144245, 0.70705632, 0.70255353, 0.70394202, 0.70389736])
duties = np.array([
     5, 10, 15, 20, 25, 30, 35, 40, 45, 50,
    55, 60, 65, 70, 75, 80, 85, 90, 95, 100
])

# Sort voltages and duties by voltage for interpolation
sorted_indices = np.argsort(voltages)
voltages_sorted = voltages[sorted_indices]
duties_sorted = duties[sorted_indices]

# Interpolation function (linear by default)
interp_func = interp1d(voltages_sorted, duties_sorted, kind='linear', fill_value='extrapolate')

# Query voltages
query_voltages = [1.00, 0.90, 0.80, 0.75, 0.71, 0.70]
estimated_duties = interp_func(query_voltages)

# Print results
for v, d in zip(query_voltages, estimated_duties):
    print(f"Interpolated Duty Cycle for {v:.2f} V: {d:.2f} %")

## Voltage-to-duty mapping — alternative fit forms not carried forward

### Exponential + spline fit comparison

In [ ]:
# Re-import necessary modules after kernel reset
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline

# Voltage and Duty arrays from your data
voltages = np.array([1.00235314, 0.82740628, 0.81370782, 0.81502292, 0.81771553,
                     0.8102661, 0.80432675, 0.79196567, 0.77636817, 0.76105513,
                     0.74830033, 0.73875982, 0.72980107, 0.72330292, 0.71230216,
                     0.71144245, 0.70705632, 0.70255353, 0.70394202, 0.70389736])
duties = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50,
                   55, 60, 65, 70, 75, 80, 85, 90, 95, 100])

# Sort for fitting stability
sorted_indices = np.argsort(voltages)
voltages = voltages[sorted_indices]
duties = duties[sorted_indices]

# 1. Exponential Fit: Duty = a * exp(b * Voltage) + c
def exp_model(v, a, b, c):
    return a * np.exp(b * v) + c

In [ ]:
exp_params, _ = curve_fit(exp_model, voltages, duties, maxfev=10000)
duty_exp_fit = exp_model(voltages, *exp_params)

# 2. Spline Fit
spline = UnivariateSpline(voltages, duties, k=3, s=0)
duty_spline_fit = spline(voltages)

# Generate fine x values for smooth curves
v_fit = np.linspace(min(voltages), max(voltages), 300)
d_exp = exp_model(v_fit, *exp_params)
d_spline = spline(v_fit)

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(voltages, duties, 'ro', label='Actual Data')
plt.plot(v_fit, d_exp, 'b-', label='Exponential Fit')
plt.plot(v_fit, d_spline, 'g--', label='Spline Fit')
plt.xlabel('Voltage (V)')
plt.ylabel('Duty Cycle (%)')
plt.title('Duty Cycle vs Voltage (Non-Polynomial Fits)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

(exp_params, spline)  # Return fitted model parameters for reference

### Exponential fit, split across three cells (kernel-reset re-import)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

voltages = np.array([1.00235314, 0.82740628, 0.81370782, 0.81502292, 0.81771553,
                     0.8102661, 0.80432675, 0.79196567, 0.77636817, 0.76105513,
                     0.74830033, 0.73875982, 0.72980107, 0.72330292, 0.71230216,
                     0.71144245, 0.70705632, 0.70255353, 0.70394202, 0.70389736])
duties = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50,
                   55, 60, 65, 70, 75, 80, 85, 90, 95, 100])

In [ ]:
def exp_model(x, a, b, c):
    return a * np.exp(b * x) + c

params, _ = curve_fit(exp_model, voltages, duties, p0=(300, -10, 0))
a, b, c = params

In [ ]:
print(f"Exponential Fit: Duty = {a:.2f} * exp({b:.2f} * Voltage) + {c:.2f}")
# Generate smooth voltage range for curve plotting
v_fit = np.linspace(min(voltages), max(voltages), 200)
d_fit = exp_model(v_fit, *params)

# Plot original data and fitted curve
plt.figure(figsize=(10, 5))
plt.plot(voltages, duties, 'ro', label='Original Data')
plt.plot(v_fit, d_fit, 'b-', label='Exponential Fit')
plt.xlabel('Voltage (V)')
plt.ylabel('Duty Cycle (%)')
plt.title('Exponential Fit: Duty Cycle vs Voltage')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Earlier (untuned) exponential fit

Superseded by the tuned version kept in notebook 05.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Your data
voltages = np.array([1.002, 0.827, 0.814, 0.815, 0.818,
                     0.810, 0.804, 0.792, 0.776, 0.761,
                     0.748, 0.739, 0.730, 0.723, 0.712,
                     0.711, 0.707, 0.703, 0.704, 0.704])
duties = np.array([5, 10, 15, 20, 25,
                   30, 35, 40, 45, 50,
                   55, 60, 65, 70, 75,
                   80, 85, 90, 95, 100])

# Define model: Exponential decay + offset
def exp_model(v, A, B, C):
    return A * np.exp(-B * v) + C

# Initial guess: [Amplitude, Decay rate, Offset]
initial_guess = [300, 10, 0]

# Fit the curve
params, _ = curve_fit(exp_model, voltages, duties, p0=initial_guess)
A_fit, B_fit, C_fit = params

# Print the fitted equation
print(f"Fitted Exponential Equation:\nDuty = {A_fit:.2f} * exp(-{B_fit:.2f} * V) + {C_fit:.2f}")

# Plotting
v_fit = np.linspace(min(voltages), max(voltages), 200)
d_fit = exp_model(v_fit, *params)

plt.figure(figsize=(10, 5))
plt.plot(voltages, duties, 'ro', label='Data')
plt.plot(v_fit, d_fit, 'b-', label='Fitted Curve')
plt.xlabel('Voltage (V)')
plt.ylabel('Duty Cycle (%)')
plt.title('Tuned Exponential Fit')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Piecewise (two-region) exponential fit

An alternative approach that was not carried forward into the final single-equation model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Your averaged data
voltages = np.array([1.002, 0.827, 0.814, 0.815, 0.818,
                     0.810, 0.804, 0.792, 0.776, 0.761,
                     0.748, 0.739, 0.730, 0.723, 0.712,
                     0.711, 0.707, 0.703, 0.704, 0.704])
duties = np.array([5, 10, 15, 20, 25,
                   30, 35, 40, 45, 50,
                   55, 60, 65, 70, 75,
                   80, 85, 90, 95, 100])

# Exponential function to fit
def exp_func(v, A, B, C):
    return A * np.exp(-B * v) + C

# Split based on duty cycle (≈50%)
split_index = np.argmax(duties > 50)
v1, d1 = voltages[:split_index], duties[:split_index]
v2, d2 = voltages[split_index:], duties[split_index:]

# Fit region 1
params1, _ = curve_fit(exp_func, v1, d1, p0=[300, 10, 0])
A1, B1, C1 = params1

# Fit region 2
params2, _ = curve_fit(exp_func, v2, d2, p0=[100, 2, 20], maxfev=5000)

A2, B2, C2 = params2

# Generate smooth curves
v_fit1 = np.linspace(min(v1), max(v1), 100)
d_fit1 = exp_func(v_fit1, *params1)

v_fit2 = np.linspace(min(v2), max(v2), 100)
d_fit2 = exp_func(v_fit2, *params2)

# Plot
plt.figure(figsize=(10, 5))
plt.scatter(voltages, duties, color='black', label='Data')
plt.plot(v_fit1, d_fit1, 'r-', label='Region 1 Exponential Fit')
plt.plot(v_fit2, d_fit2, 'b-', label='Region 2 Exponential Fit')
plt.xlabel('Voltage (V)')
plt.ylabel('Duty Cycle (%)')
plt.title('Piecewise Exponential Fit: Duty Cycle vs Voltage')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print equations
print(f"Region 1: Duty = {A1:.2f} * exp(-{B1:.2f} * V) + {C1:.2f}")
print(f"Region 2: Duty = {A2:.2f} * exp(-{B2:.2f} * V) + {C2:.2f}")

## Voltage-to-duty error-analysis pipeline — earlier (unaveraged) version

Superseded by the cleaner averaged-data pipeline (cell 34) kept in notebook 05, which produces `error_analysis_avg_voltage.csv` used downstream.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Load data
file_path = "z=2,r=0.xlsx"
df = pd.read_excel(file_path, sheet_name='21')

# Clean column names
df.columns = df.columns.str.strip()

# Drop rows with missing values
df = df.dropna(subset=['Voltage', 'Duty'])

# Extract voltage and duty as numpy arrays
voltages = df['Voltage'].values
actual_duty = df['Duty'].values

# Exponential model: Duty = 312233.9 * exp(-11.56 * V) - 0.65
predicted_duty = 312233.9 * np.exp(-11.56 * voltages) - 0.65

# Calculate error
error = predicted_duty - actual_duty

# Append results to DataFrame
df_result = pd.DataFrame({
    'Voltage': voltages,
    'Actual Duty': actual_duty,
    'Predicted Duty': predicted_duty,
    'Error': error
})

# Compute RMSE and R²
rmse = np.sqrt(mean_squared_error(actual_duty, predicted_duty))
r2 = r2_score(actual_duty, predicted_duty)

# Print metrics
print(df_result)
print(f"\nRMSE = {rmse:.4f}")
print(f"R² = {r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Load data
file_path = "z=2,r=0.xlsx"
df = pd.read_excel(file_path, sheet_name='21')

# Clean column names
df.columns = df.columns.str.strip()

# Drop rows with missing values
df = df.dropna(subset=['Voltage', 'Duty'])

# Extract voltage and duty as numpy arrays
voltages = df['Voltage'].values
actual_duty = df['Duty'].values

# Exponential model: Duty = 312233.9 * exp(-11.56 * V) - 0.65
predicted_duty = 312233.9 * np.exp(-11.56 * voltages) - 0.65

# Calculate error
error = predicted_duty - actual_duty

# Append results to DataFrame
df_result = pd.DataFrame({
    'Voltage': voltages,
    'Actual Duty': actual_duty,
    'Predicted Duty': predicted_duty,
    'Error': error
})

# Save to CSV
df_result.to_csv("predicted_vs_actual_duty.csv", index=False)

# Compute RMSE and R²
rmse = np.sqrt(mean_squared_error(actual_duty, predicted_duty))
r2 = r2_score(actual_duty, predicted_duty)

# Print metrics
print(df_result)
print(f"\nRMSE = {rmse:.4f}")
print(f"R² = {r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the CSV file
df = pd.read_csv("predicted_vs_actual_duty.csv")

# Extract actual and predicted duty values
actual = df['Actual Duty'].values
predicted = df['Predicted Duty'].values
error = df['Error'].values  # Already available from CSV

# Metrics
rmse = np.sqrt(mean_squared_error(actual, predicted))
mae = mean_absolute_error(actual, predicted)
r2 = r2_score(actual, predicted)
max_error = np.max(np.abs(error))

# Print metrics
print(f"Error Metrics:")
print(f"RMSE         = {rmse:.4f}")
print(f"MAE          = {mae:.4f}")
print(f"R² Score     = {r2:.4f}")
print(f"Max Abs Error = {max_error:.4f}")

In [ ]:
df['% Error'] = 100 * (df['Error'] / df['Actual Duty']).abs()

# Plot: Error vs Voltage
plt.figure(figsize=(10, 5))
plt.plot(df['Voltage'], df['Error'], marker='o', linestyle='-', color='red', label='Absolute Error')
plt.xlabel('Voltage (V)')
plt.ylabel('Error (Predicted - Actual) in Duty Cycle (%)')
plt.title('Error Trend vs Voltage')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Plot: % Error
plt.figure(figsize=(10, 5))
plt.plot(df['Voltage'], df['% Error'], marker='s', linestyle='--', color='purple', label='Percentage Error')
plt.xlabel('Voltage (V)')
plt.ylabel('Percentage Error (%)')
plt.title('Percentage Error vs Voltage')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Refit / model-comparison — earlier versions

Superseded by the final refit (cell 36) and comparison (cell 39) kept in notebook 05.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the dataset
df = pd.read_csv("error_analysis_avg_voltage.csv")

# Filter for voltage range 0.7V to 1.0V
df_trimmed = df[(df['Voltage'] >= 0.7) & (df['Voltage'] <= 1.0)]

V = df_trimmed['Voltage'].values
D = df_trimmed['Duty'].values  # Actual duty cycle

# === Exponential Fit ===
def exp_model(x, a, b, c):
    return a * np.exp(b * x) + c

params_exp, _ = curve_fit(exp_model, V, D, p0=(1, -1, 1))

# === Polynomial Fit (2nd order) ===
poly_coeffs = np.polyfit(V, D, 2)
poly_model = np.poly1d(poly_coeffs)

# Trying log linear
valid = (V > 0) & (D > 0)
log_V = np.log(V[valid])
log_coeffs = np.polyfit(log_V, D[valid], 1)
log_model = lambda x: log_coeffs[0] * np.log(x) + log_coeffs[1]

# === Plotting ===
voltage_range = np.linspace(0.7, 1.0, 300)

plt.figure(figsize=(10, 6))
plt.scatter(V, D, color='black', label='Actual Data')

plt.plot(voltage_range, exp_model(voltage_range, *params_exp), 'r-', label='Exponential Fit')
plt.plot(voltage_range, poly_model(voltage_range), 'b--', label='Polynomial Fit')
plt.plot(voltage_range, log_model(voltage_range), 'g-.', label='Log-Linear Fit')

plt.xlabel("Voltage (V)")
plt.ylabel("Duty Cycle (%)")
plt.title("Refit Models (0.7V–1.0V)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# === Print fitted equations ===
print(f"Exponential Fit: Duty = {params_exp[0]:.2f} * exp({params_exp[1]:.2f} * V) + {params_exp[2]:.2f}")
print(f"Polynomial Fit: Duty = {poly_coeffs[0]:.2f} * V² + {poly_coeffs[1]:.2f} * V + {poly_coeffs[2]:.2f}")
print(f"Log-Linear Fit: Duty = {log_coeffs[0]:.2f} * log(V) + {log_coeffs[1]:.2f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Load your CSV file
df = pd.read_csv("error_analysis_avg_voltage.csv")

# Use correct column names
voltages = df["Voltage"]
actual_duty = df["Duty"]

# Define the two models
def model1(v):
    return 312233.9 * np.exp(-11.56 * v) - 0.65

def model2(v):
    return 15978.86 * np.exp(-6.90 * v) - 34.73

# Apply both models
pred1 = model1(voltages)
pred2 = model2(voltages)

# Compute metrics
rmse1 = np.sqrt(mean_squared_error(actual_duty, pred1))
r2_1 = r2_score(actual_duty, pred1)

rmse2 = np.sqrt(mean_squared_error(actual_duty, pred2))
r2_2 = r2_score(actual_duty, pred2)

print(f"Model 1 (312233.9 * exp(-11.56V) - 0.65): RMSE = {rmse1:.4f}, R² = {r2_1:.4f}")
print(f"Model 2 (15978.86 * exp(-6.90V) - 34.73): RMSE = {rmse2:.4f}, R² = {r2_2:.4f}")

## Synthetic voltage profile — visualisation only

Generates the same smooth/stepped voltage profiles used in notebook 05's final application cell, but without converting them to duty cycle. Kept here as the precursor step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------- Graph 1: Smooth Peaks -------------
# Time steps
t1 = np.linspace(0, 10, 50)   # 0 to 10s: rise
t2 = np.linspace(10, 20, 50)   # 10 to 15s: fall
t3 = np.linspace(20, 30, 50)   # 15 to 20s: rise
t4 = np.linspace(30, 40, 50)   # 20 to 25s: fall

# Voltages
v1 = np.linspace(0.77, 0.85, len(t1))  # rise
v2 = np.linspace(0.85, 0.77, len(t2))  # fall
v3 = np.linspace(0.77, 0.85, len(t3))  # rise
v4 = np.linspace(0.85, 0.77, len(t4))  # fall

# Combine
time1 = np.concatenate([t1, t2, t3, t4])
volt1 = np.concatenate([v1, v2, v3, v4])

# ----------- Graph 2: Stepped Plateau -------------
# Define steps
voltage_steps = [0.77, 0.78, 0.79, 0.80, 0.79, 0.78, 0.77]
durations = [10, 10, 10, 10, 10, 10, 10]  # durations in seconds

# Build time and voltage arrays
time2 = []
volt2 = []
current_time = 0

for v, d in zip(voltage_steps, durations):
    t = np.linspace(current_time, current_time + d, d * 10)  # 10 samples per second
    time2.extend(t)
    volt2.extend([v] * len(t))
    current_time += d

# ----------- Plotting -------------
fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Graph 1
axs[0].plot(time1, volt1, color='blue')
axs[0].set_title("Graph 1: Smooth Voltage Peaks")
axs[0].set_ylabel("Voltage (V)")
axs[0].grid(True)

# Graph 2
axs[1].step(time2, volt2, where='post', color='darkgreen')
axs[1].set_title("Graph 2: Stepped Voltage Plateau")
axs[1].set_xlabel("Time (s)")
axs[1].set_ylabel("Voltage (V)")
axs[1].grid(True)

plt.tight_layout()
plt.show()

## Fan-log parsing — earlier iterations

Three earlier stages of the parsing pipeline that was finalised in notebook 04 (cell 46): basic averaging, then up/down ramp detection, then filtering + styling — each fully superseded by the next.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- Read CSV as raw text ---
with open("/content/fan_log.csv", "r", encoding="utf-16") as f:
    lines = f.readlines()

# --- Extract numeric duty and rpm values ---
data_lines = []
for line in lines:
    line = line.strip()
    if line == "" or line.startswith("---") or "Available" in line or "Duty" in line:
        continue
    # Keep only digits, '.' and ','
    line_clean = ''.join(c if c.isdigit() or c in [',', '.'] else '' for c in line)
    if ',' in line_clean:
        duty_str, rpm_str = line_clean.split(',', 1)
        try:
            duty = float(duty_str)
            rpm = float(rpm_str)
            data_lines.append([duty, rpm])
        except ValueError:
            continue

df = pd.DataFrame(data_lines, columns=["duty", "rpm"])

# --- Average RPM for each unique duty cycle ---
avg_df = df.groupby("duty", as_index=False).mean()

# --- Quick look ---
print(avg_df.head())
print(avg_df.tail())

# --- Plot ---
plt.figure(figsize=(17,14))
plt.plot(avg_df["duty"], avg_df["rpm"], label="Average RPM per Duty", linewidth=2, marker='o')
plt.xlabel("Duty Cycle (%)")
plt.ylabel("RPM")
plt.title("Duty Cycle vs RPM (Averaged per Duty Cycle)")
plt.grid(True)
plt.legend()

# --- Set x-axis ticks every 1% ---
plt.xticks(np.arange(0, int(avg_df["duty"].max()) + 2, 2))
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Read CSV as raw text ---
with open("/content/fan_log.csv", "r", encoding="utf-16") as f:
    lines = f.readlines()

# --- Extract numeric duty and rpm values ---
data_lines = []
for line in lines:
    line = line.strip()
    if line == "" or line.startswith("---") or "Available" in line or "Duty" in line:
        continue
    line_clean = ''.join(c if c.isdigit() or c in [',', '.'] else '' for c in line)
    if ',' in line_clean:
        duty_str, rpm_str = line_clean.split(',', 1)
        try:
            duty = float(duty_str)
            rpm = float(rpm_str)
            data_lines.append([duty, rpm])
        except ValueError:
            continue

df = pd.DataFrame(data_lines, columns=["duty", "rpm"])

# --- Detect up-ramp and down-ramp ---
df['ramp'] = 'up'  # default
turning_index = df['duty'].idxmax()  # assume max duty is the peak
df.loc[turning_index+1:, 'ramp'] = 'down'  # everything after peak = down

# --- Average per duty per ramp ---
avg_df = df.groupby(['ramp','duty'], as_index=False).mean()

# --- Plot ---
plt.figure(figsize=(8,5))
for ramp_type, color in zip(['up','down'], ['blue','red']):
    ramp_data = avg_df[avg_df['ramp']==ramp_type]
    plt.plot(ramp_data['duty'], ramp_data['rpm'], label=f'{ramp_type.capitalize()} Ramp', linewidth=2, marker='o')

plt.xlabel("Duty Cycle (%)")
plt.ylabel("RPM")
plt.title("Duty Cycle vs RPM")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Read CSV as raw text (handles weird encoding + junk lines) ---
with open("/content/fan_log.csv", "r", encoding="utf-16") as f:
    lines = f.readlines()

# --- Extract numeric duty and rpm values ---
data = []
for line in lines:
    line = line.strip()

    # Skip garbage
    if (
        line == "" or
        line.startswith("---") or
        "Available" in line or
        "Duty" in line
    ):
        continue

    # Keep only digits, comma, decimal
    cleaned = ''.join(c if c.isdigit() or c in [',', '.'] else '' for c in line)

    if ',' in cleaned:
        try:
            duty, rpm = map(float, cleaned.split(',', 1))
            data.append([duty, rpm])
        except ValueError:
            pass

df = pd.DataFrame(data, columns=["duty", "rpm"])

# --- Keep ONLY multiples of 10 (0, 10, 20, ...) ---
df = df[df['duty'] % 10 == 0]

# --- HARD RPM CUTOFF (physical limit) ---
df = df[df['rpm'] <= 1400]

# --- Detect up-ramp and down-ramp ---
df['ramp'] = 'up'
peak_idx = df['duty'].idxmax()
df.loc[peak_idx + 1:, 'ramp'] = 'down'

# --- Average RPM per duty per ramp ---
avg_df = (
    df
    .groupby(['ramp', 'duty'], as_index=False)
    .mean()
    .sort_values('duty')
)

# --- Plot ---
plt.figure(figsize=(8, 5))

for ramp in ['up', 'down']:
    subset = avg_df[avg_df['ramp'] == ramp]
    plt.plot(
        subset['duty'],
        subset['rpm'],
        marker='o',
        linewidth=2,
        label=f"{ramp.capitalize()} Ramp"
    )

# --- Axis formatting ---
plt.xticks(range(0, 101, 10), fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')

plt.xlabel("Duty Cycle (%)", fontsize=16, fontweight='bold')
plt.ylabel("RPM", fontsize=16, fontweight='bold')
plt.title("Duty Cycle vs RPM", fontsize=18, fontweight='bold')

plt.grid(True)
plt.legend(fontsize=13)

plt.tight_layout()
plt.show()

## Repeated-trial near-duplicates

These two cells analyse the *identical* raw dataset that is analysed cleanly in notebook 06 (Trial C, cells 54–55) — kept here only because they show slightly different plotting breakdowns.

In [ ]:
# This script:
# 1) Stores the raw RPM samples you provided
# 2) Plots RAW DATA (scatter: duty vs all RPM samples)
# 3) Computes mean RPM per duty cycle
# 4) Fits a LINEAR model (least squares)
# 5) Plots LINEAR FIT (mean RPM vs duty)

import numpy as np
import matplotlib.pyplot as plt

# ---------------- RAW DATA ----------------
data = {
    0:  [1320,90,0,0,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,60,30,0,0,0,0,0,0,0,0,0,60,0,0,0,0,30,0,0,30,0,0,30,0],
    10: [0,210,450,330,240,210,150,180,150,150,150,150,120,180,150,150,150,180,150,120,150,150,150,180,120,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,120,150,150,150,150,150,150,120,150,150,150,150,150,150,150,120,150,150,150],
    20: [150,180,270,300,330,330,360,330,360,330,360,330,360,330,360,360,360,330,360,330,360,330,330,390,330,360,330,360,330,360,330,360,330,360,330,360,330,360,330,330,360,330,360,330,360,330,360,330,360,330,360,330,330,360,330,360,330,360,330,360],
    30: [330,390,480,480,510,510,540,510,510,540,510,510,540,510,540,510,510,540,510,540,510,510,540,510,510,540,510,510,570,510,540,510,510,540,510,510,540,510,510,540,510,510,540,510,510,540,510,510,540,510,540,510,510,540,510,510,540,510,510,540],
    40: [510,570,630,660,660,660,690,660,660,690,660,660,690,660,690,660,690,660,660,690,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,660,690],
    50: [660,720,750,780,810,780,810,780,810,780,810,780,810,780,810,810,780,810,780,810,780,810,780,810,780,810,780,780,810,780,810,810,780,810,780,810,780,810,780,810,780,810,780,810,780,810,780,810,780,780,810,780,810,780,810,780,810,780,780,810],
    60: [780,840,900,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,930,900,930,930,900,930,900,930,930,930,900,930,930,900,930,930],
    70: [900,960,1020,1020,1050,1020,1020,1050,1020,1050,1020,1020,1050,1020,1020,1050,1020,1050,1020,1020,1050,1020,1020,1050,1020,1050,1020,1020,1050,1020,1050,1020,1050,1020,1050,1020,1020,1050,1020,1020,1050,1020,1050,1020,1050,1020,1020,1050,1020,1020,1050,1020,1050,1020,1050,1020,1020,1050,1020,1020],
    80: [1050,1050,1140,1110,1140,1140,1140,1140,1140,1140,1110,1140,1140,1140,1140,1140,1110,1140,1140,1140,1140,1140,1110,1140,1140,1140,1140,1140,1140,1110,1140,1140,1140,1140,1140,1110,1140,1140,1140,1110,1140,1140,1140,1140,1110,1140,1140,1140,1110,1140,1140,1140,1110,1140,1140,1140,1140,1140,1110,1140],
    90: [1140,1170,1200,1230,1200,1230,1230,1230,1230,1200,1230,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200,1230,1230,1230,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200,1230,1230,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200],
    100:[1230,1260,1290,1320,1290,1320,1320,1320,1290,1320,1320,1320,1320,1290,1320,1320,1290,1320,1320,1290,1320,1320,1320,1320,1290,1320,1320,1320,1320,1320,1290,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1350,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320]
}

# ---------------- RAW DATA PLOT ----------------
plt.figure()
for duty, rpms in data.items():
    plt.scatter([duty]*len(rpms), rpms)
plt.xlabel("Duty Cycle (%)")
plt.ylabel("RPM")
plt.title("Raw RPM Data vs Duty Cycle")
plt.grid(True)
plt.show()

# ---------------- MEAN + LINEAR FIT ----------------
duties = np.array(sorted(data.keys()))
mean_rpm = np.array([np.mean(data[d]) for d in duties])

# Linear fit
coeffs = np.polyfit(duties, mean_rpm, 1)
fit_fn = np.poly1d(coeffs)
fit_rpm = fit_fn(duties)

plt.figure()
plt.plot(duties, mean_rpm, marker='o', label="Mean RPM")
plt.plot(duties, fit_rpm, linestyle='--', label="Linear Fit")
plt.xlabel("Duty Cycle (%)")
plt.ylabel("RPM")
plt.title("Duty Cycle vs RPM (Linear Fit)")
plt.legend()
plt.grid(True)
plt.show()

coeffs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------- RAW DATA ----------------
data = {
    0:  [1320,90,0,0,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,60,30,0,0,0,0,0,0,0,0,0,60,0,0,0,0,30,0,0,30,0,0,30,0],
    10: [0,210,450,330,240,210,150,180,150,150,150,150,120,180,150,150,150,180,150,120,150,150,150,180,120,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,120,150,150,150,150,150,150,120,150,150,150,150,150,150,150,120,150,150,150],
    20: [150,180,270,300,330,330,360,330,360,330,360,330,360,330,360,360,360,330,360,330,360,330,330,390,330,360,330,360,330,360,330,360,330,360,330,360,330,360,330,330,360,330,360,330,360,330,360,330,360,330,360,330,330,360,330,360,330,360,330,360],
    30: [330,390,480,480,510,510,540,510,510,540,510,510,540,510,540,510,510,540,510,540,510,510,540,510,510,540,510,510,570,510,540,510,510,540,510,510,540,510,510,540,510,510,540,510,510,540,510,510,540,510,540,510,510,540,510,510,540,510,510,540],
    40: [510,570,630,660,660,660,690,660,660,690,660,660,690,660,690,660,690,660,660,690,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,690,660,660,690,660,660,690],
    50: [660,720,750,780,810,780,810,780,810,780,810,780,810,780,810,810,780,810,780,810,780,810,780,810,780,810,780,780,810,780,810,810,780,810,780,810,780,810,780,810,780,810,780,810,780,810,780,810,780,780,810,780,810,780,810,780,810,780,780,810],
    60: [780,840,900,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,930,900,930,930,900,930,930,900,930,930,900,930,930,900,930,930,930,900,930,930,900,930,900,930,930,930,900,930,930,900,930,930],
    70: [900,960,1020,1020,1050,1020,1020,1050,1020,1050,1020,1020,1050,1020,1020,1050,1020,1050,1020,1020,1050,1020,1020,1050,1020,1050,1020,1020,1050,1020,1050,1020,1050,1020,1050,1020,1020,1050,1020,1020,1050,1020,1050,1020,1050,1020,1020,1050,1020,1020,1050,1020,1050,1020,1050,1020,1020,1050,1020,1020],
    80: [1050,1050,1140,1110,1140,1140,1140,1140,1140,1140,1110,1140,1140,1140,1140,1140,1110,1140,1140,1140,1140,1140,1110,1140,1140,1140,1140,1140,1140,1110,1140,1140,1140,1140,1140,1110,1140,1140,1140,1110,1140,1140,1140,1140,1110,1140,1140,1140,1110,1140,1140,1140,1110,1140,1140,1140,1140,1140,1110,1140],
    90: [1140,1170,1200,1230,1200,1230,1230,1230,1230,1200,1230,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200,1230,1230,1230,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200,1230,1230,1230,1230,1200,1230,1230,1230,1200,1230,1230,1200],
    100:[1230,1260,1290,1320,1290,1320,1320,1320,1290,1320,1320,1320,1320,1290,1320,1320,1290,1320,1320,1290,1320,1320,1320,1320,1290,1320,1320,1320,1320,1320,1290,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1350,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320,1320]
}

# ---------------- 1) RAW SCATTER ----------------
plt.figure()
for duty, rpms in data.items():
    plt.scatter([duty]*len(rpms), rpms, alpha=0.6)
plt.xlabel("Duty Cycle (%)")
plt.ylabel("RPM")
plt.title("Raw RPM Samples vs Duty Cycle")
plt.xticks(np.arange(0, 101, 10))
plt.grid(True)
plt.show()

# ---------------- MEAN DATA ----------------
duties = np.array(sorted(data.keys()))
mean_rpm = np.array([np.mean(data[d]) for d in duties])

# ---------------- 2) MEAN ONLY ----------------
plt.figure()
plt.plot(duties, mean_rpm, marker='o')
plt.xlabel("Duty Cycle (%)")
plt.ylabel("Mean RPM")
plt.title("Mean RPM vs Duty Cycle")
plt.xticks(np.arange(0, 101, 10))
plt.grid(True)
plt.show()

# ---------------- 3) LINEAR FIT ONLY ----------------
coeffs = np.polyfit(duties, mean_rpm, 1)
fit_fn = np.poly1d(coeffs)
fit_rpm = fit_fn(duties)

plt.figure()
plt.plot(duties, fit_rpm, linestyle='--')
plt.xlabel("Duty Cycle (%)")
plt.ylabel("RPM (Linear Model)")
plt.title("Linear Fit: RPM vs Duty Cycle")
plt.xticks(np.arange(0, 101, 10))
plt.grid(True)
plt.show()

coeffs